# Regression detection for AI agents — Phoenix + Kalibra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khan5v/kalibra/blob/main/examples/phoenix_kalibra_tutorial.ipynb)

**The experiment:** A 3-step agent (plan → tool → respond) runs 25 tasks — 12 simple, 13 complex. The baseline gives the respond step `max_tokens=1024`. The "current" build cuts it to `max_tokens=150`. Same model, same prompts, one parameter change.

**What happens:** Aggregate metrics *improve* — fewer tokens, lower cost, faster. But complex tasks are silently truncated. The top-line numbers say "ship it." The per-task breakdown says "don't."

**Stack:** [Phoenix](https://github.com/Arize-ai/phoenix) (trace collection) → [OpenInference](https://github.com/Arize-ai/openinference) (trace format) → [Kalibra](https://github.com/khan5v/kalibra) (statistical comparison)

**Two modes:**
- **No API key** — uses pre-recorded traces committed to the repo. All Kalibra analysis cells work identically.
- **With `ANTHROPIC_API_KEY`** — runs the agent live through Phoenix, exports fresh traces, then compares.

In [ ]:
!pip install -q kalibra

## 1. Mode detection

If `ANTHROPIC_API_KEY` is set, we run the full live flow: instrument with Phoenix, call the Anthropic API, export traces. If not, we skip the live cells and use pre-recorded traces instead. All Kalibra analysis is identical either way.

In [16]:
import os

# Colab: reads from Colab Secrets if available.
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

LIVE_MODE = bool(os.environ.get("ANTHROPIC_API_KEY"))

if LIVE_MODE:
    print("API key found — running in LIVE mode (Phoenix + Anthropic API).")
else:
    print("No API key — running in OFFLINE mode (pre-recorded traces).")
    print("Set ANTHROPIC_API_KEY to run the agent live.")

No API key — running in OFFLINE mode (pre-recorded traces).
Set ANTHROPIC_API_KEY to run the agent live.


## The tasks

25 tasks split into two groups — this is what makes the truncation story work:

- **12 simple** (translate, acronym, haiku, emoji, oneliner, boolean, convert, opposite, count, abbreviate, binary, sla) — short answers that fit in 150 tokens
- **13 complex** (sql, ztest, tradeoffs, cap, regression, refactor, review, debug, architecture, migration, pipeline, security, consensus) — multi-step reasoning that needs 300-1000+ tokens

When `max_tokens` drops to 150, the simple tasks are fine. The complex tasks truncate silently. The aggregate improves. The breakdown catches it.

## 2. Generate traces (live mode only)

*Skip this cell if running offline — it uses pre-recorded traces instead.*

With an API key, this cell starts Phoenix, builds a 3-step agent (plan → tool → respond), runs it on all 25 tasks twice per variant, and exports everything to a single JSONL file tagged with `variant == baseline` or `variant == current`.

```
agent (CHAIN)
 +-- messages.create (LLM)     <- plan step
 +-- tool:prepare_context (TOOL) <- simulated tool
 +-- messages.create (LLM)     <- respond step (truncation happens here)
```

In [17]:
if not LIVE_MODE:
    print("Skipped — using pre-recorded traces (no API key).")
    print("Jump to the next cell to load and analyze.")
else:
    # ── Install Phoenix + Anthropic deps ──────────────────────────────
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "arize-phoenix", "arize-phoenix-otel",
        "openinference-instrumentation-anthropic", "anthropic",
    ])

    # ── Start Phoenix ─────────────────────────────────────────────────
    import time, json, anthropic
    import phoenix as px
    from opentelemetry import trace
    from phoenix.otel import register
    from phoenix.client import Client
    from openinference.instrumentation.anthropic import AnthropicInstrumentor

    session = px.launch_app()
    tracer_provider = register()
    AnthropicInstrumentor().instrument(tracer_provider=tracer_provider)
    tracer = trace.get_tracer("tutorial-agent")
    llm = anthropic.Anthropic()
    phoenix_client = Client()
    MODEL = "claude-haiku-4-5-20251001"

    print(f"Phoenix UI: {session.url}")

    # ── Tasks ─────────────────────────────────────────────────────────
    TASKS = [
        {"id": "translate",   "prompt": "Translate 'good morning' to French, German, and Japanese. Just the translations."},
        {"id": "acronym",     "prompt": "What does API stand for?"},
        {"id": "haiku",       "prompt": "Write a haiku about a server crash."},
        {"id": "emoji",       "prompt": "Represent the CI/CD pipeline in emojis. One per stage."},
        {"id": "oneliner",    "prompt": "Explain what a hash table is in one sentence."},
        {"id": "boolean",     "prompt": "Is a two-proportion z-test parametric or non-parametric? One word answer."},
        {"id": "convert",     "prompt": "Convert 72F to Celsius. Show just the number."},
        {"id": "opposite",    "prompt": "What is the opposite of 'idempotent' in API design?"},
        {"id": "count",       "prompt": "How many bits in a byte?"},
        {"id": "abbreviate",  "prompt": "What does CRUD stand for in database operations?"},
        {"id": "binary",      "prompt": "What is 255 in binary?"},
        {"id": "sla",         "prompt": "What does SLA stand for in cloud computing?"},
        {"id": "sql",         "prompt": "Write a SQL query to find customers who made a purchase every month of 2024. Use table orders(customer_id, order_date, amount). Include comments explaining each part."},
        {"id": "ztest",       "prompt": "Explain the two-proportion z-test: what it tests, assumptions, formula, and a worked example with actual numbers."},
        {"id": "tradeoffs",   "prompt": "Compare microservices vs monolith. Give 3 scenarios where each wins, with concrete reasoning and examples."},
        {"id": "cap",         "prompt": "Explain the CAP theorem, why it's commonly misunderstood, and what the PACELC extension adds. Give a real-world example for each trade-off."},
        {"id": "regression",  "prompt": "Describe 3 statistical methods for detecting regressions in A/B tests. For each: how it works, when to use it, when it fails, and a code sketch."},
        {"id": "refactor",    "prompt": "Show how to refactor a 50-line Python function that mixes I/O, validation, and business logic into clean, testable components. Show before and after code."},
        {"id": "review",      "prompt": "Write a thorough code review for this function:\ndef process(data):\n  results = []\n  for item in data:\n    if item.get('status') == 'active' and item.get('score', 0) > 0.5:\n      results.append({'id': item['id'], 'value': item['score'] * 100})\n  return sorted(results, key=lambda x: -x['value'])[:10]"},
        {"id": "debug",       "prompt": "A Python web server returns 502 errors under load. Walk through a systematic debugging process: what to check first, what tools to use, common root causes, and how to fix each one."},
        {"id": "architecture","prompt": "Design a system that processes 10,000 webhook events per second with at-least-once delivery. Cover: queue choice, consumer design, retry strategy, dead letter handling, and monitoring."},
        {"id": "migration",   "prompt": "Plan a zero-downtime migration from PostgreSQL to a new schema that renames 5 columns, splits one table into two, and adds a new index. Include rollback strategy."},
        {"id": "pipeline",    "prompt": "Design a data pipeline that ingests 1M CSV rows daily from 3 sources with different schemas, validates, deduplicates, transforms into a star schema, and loads into a data warehouse. Cover error handling, monitoring, and backfill strategy."},
        {"id": "security",    "prompt": "Explain the OWASP top 3 web vulnerabilities. For each: what it is, a concrete code example showing the vulnerability, how to exploit it, and the fix. Use Python/Flask examples."},
        {"id": "consensus",   "prompt": "Compare Raft and Paxos consensus algorithms. For each: how leader election works, how log replication works, what happens during network partitions, and when to choose one over the other. Include pseudocode."},
    ]

    # ── Agent ─────────────────────────────────────────────────────────
    def run_agent(task, max_tokens_respond=1024, run_id=0, variant="baseline"):
        with tracer.start_as_current_span("agent", attributes={
            "openinference.span.kind": "CHAIN",
            "task.id": task["id"], "task.run_id": run_id, "variant": variant,
        }):
            plan = llm.messages.create(
                model=MODEL, max_tokens=200,
                messages=[{"role": "user", "content":
                    f"In one sentence, describe how you would approach this task: {task['prompt']}"}],
            ).content[0].text

            with tracer.start_as_current_span("tool:prepare_context", attributes={
                "openinference.span.kind": "TOOL",
                "tool.name": "prepare_context",
            }):
                time.sleep(0.1)
                context = f"Approach: {plan[:200]}"

            response = llm.messages.create(
                model=MODEL, max_tokens=max_tokens_respond,
                messages=[
                    {"role": "user", "content": task["prompt"]},
                    {"role": "assistant", "content": context},
                    {"role": "user", "content": "Continue with the full response."},
                ],
            )
        return response.stop_reason

    # ── Baseline run ──────────────────────────────────────────────────
    print(f"\nBaseline (max_tokens=1024)...")
    for run_id in range(2):
        for task in TASKS:
            stop = run_agent(task, max_tokens_respond=1024, run_id=run_id, variant="baseline")
            tag = " [TRUNCATED]" if stop == "max_tokens" else ""
            print(f"  [{task['id']}-run{run_id}]{tag}")

    # ── Current run ───────────────────────────────────────────────────
    print(f"\nCurrent (max_tokens=150)...")
    for run_id in range(2):
        for task in TASKS:
            stop = run_agent(task, max_tokens_respond=150, run_id=run_id, variant="current")
            tag = " [TRUNCATED]" if stop == "max_tokens" else ""
            print(f"  [{task['id']}-run{run_id}]{tag}")

    # ── Export ────────────────────────────────────────────────────────
    time.sleep(5)
    all_spans = phoenix_client.spans.get_spans(project_identifier="default")
    with open("traces_live.jsonl", "w") as f:
        for span in all_spans:
            f.write(json.dumps(span, default=str) + "\n")
    print(f"\nExported {len(all_spans)} spans -> traces_live.jsonl")

Skipped — using pre-recorded traces (no API key).
Jump to the next cell to load and analyze.


## 3. Load traces and split with `where`

Kalibra auto-detects OpenInference format. We load a single file containing both populations, then split into baseline/current using the `variant` attribute — the same `where` filtering you'd use in `kalibra.yml`.

In [18]:
import os
from pathlib import Path
from kalibra.loader import load_traces
from kalibra.config import parse_matcher

# ── Resolve trace file ────────────────────────────────────────────────
if LIVE_MODE:
    TRACES_PATH = "traces_live.jsonl"
else:
    candidates = [
        Path("examples/traces/traces.jsonl"),  # from repo root
        Path("traces/traces.jsonl"),            # relative to notebook
    ]
    TRACES_PATH = next((str(p) for p in candidates if p.exists()), None)
    if TRACES_PATH is None:
        import urllib.request
        os.makedirs("traces", exist_ok=True)
        url = "https://raw.githubusercontent.com/khan5v/kalibra/main/examples/traces/traces.jsonl"
        TRACES_PATH = "traces/traces.jsonl"
        print("Downloading pre-recorded traces...")
        urllib.request.urlretrieve(url, TRACES_PATH)

# ── Load and split ────────────────────────────────────────────────────
all_traces = load_traces(TRACES_PATH)

m_base = parse_matcher("variant == baseline")
m_curr = parse_matcher("variant == current")
baseline = [t for t in all_traces if m_base.matches(t.metadata.get(m_base.field))]
current = [t for t in all_traces if m_curr.matches(t.metadata.get(m_curr.field))]

print(f"Loaded {len(all_traces)} traces from {TRACES_PATH}")
print(f"  Baseline: {len(baseline)} traces (variant == baseline)")
print(f"  Current:  {len(current)} traces (variant == current)")
print()

# Show the tree structure of the first trace.
t = baseline[0]
print(f"Example trace ({t.trace_id[:12]}...):")
print(f"  {len(t.spans)} spans, {len(t.leaf_spans())} leaves, {t.total_tokens} tokens")
for i, s in enumerate(t.spans):
    prefix = "  +--" if i == len(t.spans) - 1 else "  |--"
    tokens = f"tokens={s.total_tokens}" if s.total_tokens else "no tokens"
    print(f"{prefix} {s.name} ({tokens})")

Loaded 100 traces from traces/traces.jsonl
  Baseline: 50 traces (variant == baseline)
  Current:  50 traces (variant == current)

Example trace (b902b2a37e0c...):
  4 spans, 3 leaves, 1405 tokens
  |-- agent (no tokens)
  |-- messages.create (tokens=266)
  |-- tool:prepare_context (no tokens)
  +-- messages.create (tokens=1139)


## 4. Compare with quality gates

Two gates that tell a story:

- `token_delta_pct <= -10` — tokens went *down*. This gate **passes**. Looks like an improvement.
- `regressions <= 2` — per-task regressions. This gate **fails**. Complex tasks broke.

The aggregate metric lies. The per-task breakdown doesn't.

### Terminal output

This is what you see when you run `kalibra compare -v` locally.

In [19]:
from kalibra.engine import compare
from kalibra.renderers import render

result = compare(
    baseline, current,
    require=[
        "token_delta_pct <= -10",  # tokens decreased — passes (misleadingly)
        "regressions <= 2",         # per-task regressions — fails (complex tasks broke)
    ],
    baseline_source="max_tokens=1024",
    current_source="max_tokens=150",
    metric_config={"trace_breakdown": {"task_id_field": "task.id"}},
)

print(render(result, "terminal", verbose=True))


  Kalibra Compare
  ──────────────────────────────────────────────────────────
  Baseline        50 traces   (max_tokens=1024)
  Current         50 traces   (max_tokens=150)
  Gates         ✗ 1/2 failed

  Trace metrics

  ▼ Success rate      50.0% → 18.0%  -32.0 pp   (p=0.001)
                      p=0.001 — statistically significant
                      n=50→50 traces with outcomes

  — Cost              n/a — no cost data
                      !  No cost data found

  ▲ Duration          7.1s → 3.3s median  -52.9%
                      6.7s → 3.5s avg  -47.3%
                      95% CI [-62.3%, -14.0%]

  ≈ Steps             3 → 3 steps/trace (median)  +0.0%
                      3.0 → 3.0 avg  +0.0%
                      95% CI [+0.0%, +0.0%]

  ≈ Error rate        0.0% → 0.0%  +0.0 pp
                      p=1.000 — not statistically significant

  ▲ Token usage       963 → 337 tokens/trace (median)  -65.0%
                      814 → 346 avg  -57.5%
                      in: 

### PR comment (Markdown)

This is what the [Kalibra GitHub Action](https://github.com/khan5v/kalibra-action) posts to your pull request.

In [20]:
from IPython.display import Markdown

Markdown(render(result, "markdown", verbose=True))

## ❌ Kalibra Compare — 1/2 quality gate failed

**Baseline:** 50 traces (max_tokens=1024)  
**Current:** 50 traces (max_tokens=150)

### Metrics

| Metric | Direction | Delta | Detail |
|--------|-----------|-------|--------|
| Success rate | ▼ | -32.0 pp | 50.0% → 18.0% |
| Cost | – | — | n/a |
| Duration | ▲ | -52.9% | 7.1s → 3.3s median |
| Steps | — | +0.0% | 3 → 3 steps |
| Error rate | — | +0.0 pp | 0.0% → 0.0% |
| Token usage | ▲ | -65.0% | 963 → 337 tokens |
| Token efficiency | ▲ | -36.5% | 299 → 190 tok/success |
| Cost / quality | – | — | n/a |

### Trace Breakdown

**10** regressions, **0** improvements

<details><summary>Regressions</summary>

- **abbreviate**: 2/2 → 0/2
- **acronym**: 2/2 → 0/2
- **binary**: 2/2 → 1/2
- **boolean**: 2/2 → 0/2
- **count**: 2/2 → 1/2
- **emoji**: 2/2 → 0/2
- **oneliner**: 2/2 → 1/2
- **opposite**: 2/2 → 0/2
- **sla**: 2/2 → 0/2
- **sql**: 1/2 → 0/2

</details>

### Span Breakdown

3 matched — **1** improved

| Span | Direction | Duration | Cost | Tokens |
|------|-----------|----------|------|--------|
| agent | ▲ | -53% | +0% | +0% |

### Quality Gates

| Gate | Result | Actual |
|------|--------|--------|
| `token_delta_pct <= -10` | ✅ PASS | -65.00 |
| `regressions <= 2` | ❌ FAIL | 10.00 |

❌ **Quality gate violation**


### CI gate check

This is the exit code your CI pipeline checks — `kalibra compare` returns exit 1 when any gate fails.

In [21]:
print(f"Direction: {result.direction.value}")
print(f"Gates passed: {result.passed}")
for gate in result.gates:
    status = "PASS" if gate.passed else "FAIL"
    print(f"  [{status}] {gate.expr}  (actual: {gate.actual})")

if not result.passed:
    print("\nThis change would be blocked in CI.")
else:
    print("\nAll gates passed — change is within budget.")

Direction: inconclusive
Gates passed: False
  [PASS] token_delta_pct <= -10  (actual: -65.0)
  [FAIL] regressions <= 2  (actual: 10.0)

This change would be blocked in CI.


## Interpreting the results

Token usage decreased. That looks like an improvement.

But check the trace breakdown — complex tasks flipped from complete responses to truncated ones. The tokens went down because responses were cut short, not because the agent got more efficient.

The `token_delta_pct <= -10` gate passed — the aggregate metric says "tokens are down, ship it." But the `regressions <= 2` gate caught the real problem: multiple tasks regressed when measured individually. In CI, this blocks the deploy.

This is a demo scenario with one clean variable. In practice, regressions are subtler — one task type breaks while another improves, and the top-line number doesn't move. That's where per-task breakdown and statistical testing earn their keep. See [Aggregate metrics are a blind spot in agent evaluation](https://orekhov.work/blog/kalibra-regression-detection/) for the harder cases.

## Next steps

- **Try different changes.** Swap models, change system prompts, adjust parameters. Same traces → same analysis.
- **Run more tasks.** More traces = tighter confidence intervals = more reliable comparisons.
- **Add stricter gates.** `duration_delta_pct <= 30` catches latency regressions. `cost_delta_pct <= 20` catches cost blowups.
- **Run in CI.** Use [`khan5v/kalibra-action`](https://github.com/khan5v/kalibra-action) to gate PRs with a Markdown report.

---

[Kalibra](https://github.com/khan5v/kalibra) · [Docs](https://kalibra.cc) · [GitHub Action](https://github.com/khan5v/kalibra-action) · [Phoenix](https://github.com/Arize-ai/phoenix)